In [1]:
!pip3 install googletrans==3.1.0a0

     |████████████████████████████████| 55 kB 1.5 MB/s 
     |████████████████████████████████| 42 kB 585 kB/s 
     |████████████████████████████████| 1.3 MB 19.6 MB/s 
     |████████████████████████████████| 53 kB 702 kB/s 
     |████████████████████████████████| 65 kB 2.0 MB/s 
  Created wheel for googletrans: filename=googletrans-3.1.0a0-py3-none-any.whl size=16367 sha256=f32a4fad7fbdd2c0af61ebf2efaafdc2eee45f021b25656ab2e3ffa2f2647b37
  Stored in directory: /root/.cache/pip/wheels/0c/be/fe/93a6a40ffe386e16089e44dad9018ebab9dc4cb9eb7eab65ae
Successfully built googletrans


In [2]:
pip install google_trans_new

In [3]:
pip install googletrans


**READ DATASET**

In [4]:
import tweepy
from textblob import TextBlob
from wordcloud import WordCloud
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
from google.colab import drive
drive.mount('drive')

Mounted at drive


In [5]:
data=pd.read_csv('/content/medictreat2.csv')
data

,date,username,fulltext
0,2022-03-08 15:30:16,Pluviophile_Arn,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣
1,2022-03-08 15:27:05,aa_anting,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...
2,2022-03-08 15:25:38,ayukaaa9,Pas lahir juga ga nangis. Yaa dokternya sibuk ...
3,2022-03-08 15:23:41,VeramitaNanda,"Trs ada lagi, pas kuliah fisiologi persarafan ..."
4,2022-03-08 15:22:31,recycledump,dokternya bilang gigiku cantik tapi sambil ket...
...,...,...,...
1742,2022-03-04 05:45:33,boredafhere,@HalodocID Ini kenapa sy mau consult dokter ma...
1743,2022-03-04 05:42:43,Gcella_,@mufazahd @kopites_94 Emang mau dokternya?
1744,2022-03-04 05:42:42,evankurkur,"Habis ke dokter THT, Expetasinya pengen di ber..."
1745,2022-03-04 05:42:41,rizz_haitami,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k..."


**Delete The Column We Don't Need**

In [6]:
data.drop(['date','username'], axis=1, inplace=True)
data.rename(columns={'fulltext':'Tweets'})

,Tweets
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...
3,"Trs ada lagi, pas kuliah fisiologi persarafan ..."
4,dokternya bilang gigiku cantik tapi sambil ket...
...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...
1743,@mufazahd @kopites_94 Emang mau dokternya?
1744,"Habis ke dokter THT, Expetasinya pengen di ber..."
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k..."


**TAHAP PREPROCESSING**

1. Delete Username in Text

In [7]:
# remove user
def remove_user(input_txt, pattern):
    r = re.findall(pattern, input_txt)
    for i in r:
        input_txt = re.sub(i, '', input_txt)
    return input_txt    
data['tweet_remove_user'] = np.vectorize(remove_user)(data['fulltext'], "@[\w]*")
data

,fulltext,tweet_remove_user
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh🤣
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...,Bukan mksdnya yg ke ruang icu nanyain kondisi...
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis. Yaa dokternya sibuk ...
3,"Trs ada lagi, pas kuliah fisiologi persarafan ...","Trs ada lagi, pas kuliah fisiologi persarafan ..."
4,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ket...
...,...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...,Ini kenapa sy mau consult dokter malah ga bis...
1743,@mufazahd @kopites_94 Emang mau dokternya?,Emang mau dokternya?
1744,"Habis ke dokter THT, Expetasinya pengen di ber...","Habis ke dokter THT, Expetasinya pengen di ber..."
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k...","Bkn cm itu, hrsnya kamera jg jgn ngeshoot dr..."


2. Delete Emoji,Retweet,etc

In [8]:
pip install Sastrawi

     |████████████████████████████████| 209 kB 10.6 MB/s 


In [9]:
import pandas as pd
import numpy as np
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import re
import unicodedata

In [10]:
# cleaning
def cleaning(strTweet):
    #remove non-ascii
    strTweet = unicodedata.normalize('NFKD', strTweet).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    #remove URLs
    strTweet = re.sub(r'(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:\'".,<>?«»“”‘’]))', '', strTweet)
    
    #remove punctuations
    strTweet = re.sub(r'[^\w]|_',' ',strTweet)
    
    #remove digit from string
    strTweet = re.sub("\S*\d\S*", "", strTweet).strip()
    
    #remove digit or numbers
    strTweet = re.sub(r"\b\d+\b", " ", strTweet)
    
    #Remove additional white spaces
    strTweet = re.sub('[\s]+', ' ', strTweet)
    
    #remove rt
    strTweet = re.sub(r'^RT[\s]+', '', strTweet)
    
    #remove tab, new line, and backslice
    strTweet = strTweet.replace('\\t', " ").replace('\\n', " ").replace('\\u', " ").replace('\\', "")
    
    # remove whitespace leading & trailing
    strTweet = strTweet.strip()
    
    # remove multiple whitespace into single whitespace
    strTweet = re.sub('\s+',' ',strTweet)
    
    # remove single character
    strTweet = re.sub(r"\b[a-zA-Z]\b", "", strTweet)
    
    return strTweet
data['tweet_cleaning'] = data['tweet_remove_user'].apply(cleaning)
data

,fulltext,tweet_remove_user,tweet_cleaning
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...,Bukan mksdnya yg ke ruang icu nanyain kondisi...,Bukan mksdnya yg ke ruang icu nanyain kondisi ...
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis Yaa dokternya sibuk l...
3,"Trs ada lagi, pas kuliah fisiologi persarafan ...","Trs ada lagi, pas kuliah fisiologi persarafan ...",Trs ada lagi pas kuliah fisiologi persarafan k...
4,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ketawa
...,...,...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...,Ini kenapa sy mau consult dokter malah ga bis...,Ini kenapa sy mau consult dokter malah ga bisa...
1743,@mufazahd @kopites_94 Emang mau dokternya?,Emang mau dokternya?,Emang mau dokternya
1744,"Habis ke dokter THT, Expetasinya pengen di ber...","Habis ke dokter THT, Expetasinya pengen di ber...",Habis ke dokter THT Expetasinya pengen di bers...
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k...","Bkn cm itu, hrsnya kamera jg jgn ngeshoot dr...",Bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...


**TAHAP CASE FOLDING (PREPROCESSING)**

In [11]:
# casefolding
def caseFolding(s):
    newStrTweetCaseFold = s.lower()
    
    return newStrTweetCaseFold
data['tweet_case_folding'] = data['tweet_cleaning'].apply(caseFolding)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh,anjir wkwwkwkkw dokternya auto ngeuh
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...,Bukan mksdnya yg ke ruang icu nanyain kondisi...,Bukan mksdnya yg ke ruang icu nanyain kondisi ...,bukan mksdnya yg ke ruang icu nanyain kondisi ...
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis Yaa dokternya sibuk l...,pas lahir juga ga nangis yaa dokternya sibuk l...
3,"Trs ada lagi, pas kuliah fisiologi persarafan ...","Trs ada lagi, pas kuliah fisiologi persarafan ...",Trs ada lagi pas kuliah fisiologi persarafan k...,trs ada lagi pas kuliah fisiologi persarafan k...
4,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ketawa,dokternya bilang gigiku cantik tapi sambil ketawa
...,...,...,...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...,Ini kenapa sy mau consult dokter malah ga bis...,Ini kenapa sy mau consult dokter malah ga bisa...,ini kenapa sy mau consult dokter malah ga bisa...
1743,@mufazahd @kopites_94 Emang mau dokternya?,Emang mau dokternya?,Emang mau dokternya,emang mau dokternya
1744,"Habis ke dokter THT, Expetasinya pengen di ber...","Habis ke dokter THT, Expetasinya pengen di ber...",Habis ke dokter THT Expetasinya pengen di bers...,habis ke dokter tht expetasinya pengen di bers...
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k...","Bkn cm itu, hrsnya kamera jg jgn ngeshoot dr...",Bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...


**TAHAP TOKENIZING (PREPROCESSING)**

In [12]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [13]:
# tokenizing
def tweetTokenization(s):
    tokens = word_tokenize(s)

    return tokens
data['tweet_tokenization'] = data['tweet_case_folding'].apply(tweetTokenization)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh,anjir wkwwkwkkw dokternya auto ngeuh,"[anjir, wkwwkwkkw, dokternya, auto, ngeuh]"
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...,Bukan mksdnya yg ke ruang icu nanyain kondisi...,Bukan mksdnya yg ke ruang icu nanyain kondisi ...,bukan mksdnya yg ke ruang icu nanyain kondisi ...,"[bukan, mksdnya, yg, ke, ruang, icu, nanyain, ..."
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis Yaa dokternya sibuk l...,pas lahir juga ga nangis yaa dokternya sibuk l...,"[pas, lahir, juga, ga, nangis, yaa, dokternya,..."
3,"Trs ada lagi, pas kuliah fisiologi persarafan ...","Trs ada lagi, pas kuliah fisiologi persarafan ...",Trs ada lagi pas kuliah fisiologi persarafan k...,trs ada lagi pas kuliah fisiologi persarafan k...,"[trs, ada, lagi, pas, kuliah, fisiologi, persa..."
4,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ketawa,dokternya bilang gigiku cantik tapi sambil ketawa,"[dokternya, bilang, gigiku, cantik, tapi, samb..."
...,...,...,...,...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...,Ini kenapa sy mau consult dokter malah ga bis...,Ini kenapa sy mau consult dokter malah ga bisa...,ini kenapa sy mau consult dokter malah ga bisa...,"[ini, kenapa, sy, mau, consult, dokter, malah,..."
1743,@mufazahd @kopites_94 Emang mau dokternya?,Emang mau dokternya?,Emang mau dokternya,emang mau dokternya,"[emang, mau, dokternya]"
1744,"Habis ke dokter THT, Expetasinya pengen di ber...","Habis ke dokter THT, Expetasinya pengen di ber...",Habis ke dokter THT Expetasinya pengen di bers...,habis ke dokter tht expetasinya pengen di bers...,"[habis, ke, dokter, tht, expetasinya, pengen, ..."
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k...","Bkn cm itu, hrsnya kamera jg jgn ngeshoot dr...",Bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,"[bkn, cm, itu, hrsnya, kamera, jg, jgn, ngesho..."


**TAHAP STOPWORDS (PREPROCESSING)**

In [14]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [15]:
# stopwords
list_stopwords = stopwords.words('indonesian')

list_stopwords.extend(["dg", "ny","d", "klo",
                      "amp", "krn", "n", 'u', 'jd', "nyg",
                       "hehe", "nder", "der", "pen", "sis", "jg",
                       "bgt", 'dah', 'ni', 'so', 'x', 'ri', 'dos', 'eee',
                       'skrng', 'skr', 'kpd', 'j', 's', 'b', 'jgn2', 'gara2',
                       'utk', 'y', 'g', 'm', 'pm', 't', 'dm', 'rm', 'p', 'indonesi', 'https',
                       'ampe', 'rt'
                      ])

list_stopwords = set(list_stopwords)

def removeStopwords(words):
    return ' '.join([word for word in words if word not in list_stopwords])
data['tweet_stopwords'] = data['tweet_tokenization'].apply(removeStopwords)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization,tweet_stopwords
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh,anjir wkwwkwkkw dokternya auto ngeuh,"[anjir, wkwwkwkkw, dokternya, auto, ngeuh]",anjir wkwwkwkkw dokternya auto ngeuh
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...,Bukan mksdnya yg ke ruang icu nanyain kondisi...,Bukan mksdnya yg ke ruang icu nanyain kondisi ...,bukan mksdnya yg ke ruang icu nanyain kondisi ...,"[bukan, mksdnya, yg, ke, ruang, icu, nanyain, ...",mksdnya yg ruang icu nanyain kondisi andin dok...
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis Yaa dokternya sibuk l...,pas lahir juga ga nangis yaa dokternya sibuk l...,"[pas, lahir, juga, ga, nangis, yaa, dokternya,...",pas lahir ga nangis yaa dokternya sibuk lgi op...
3,"Trs ada lagi, pas kuliah fisiologi persarafan ...","Trs ada lagi, pas kuliah fisiologi persarafan ...",Trs ada lagi pas kuliah fisiologi persarafan k...,trs ada lagi pas kuliah fisiologi persarafan k...,"[trs, ada, lagi, pas, kuliah, fisiologi, persa...",trs pas kuliah fisiologi persarafan kandung ke...
4,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ketawa,dokternya bilang gigiku cantik tapi sambil ketawa,"[dokternya, bilang, gigiku, cantik, tapi, samb...",dokternya bilang gigiku cantik ketawa
...,...,...,...,...,...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...,Ini kenapa sy mau consult dokter malah ga bis...,Ini kenapa sy mau consult dokter malah ga bisa...,ini kenapa sy mau consult dokter malah ga bisa...,"[ini, kenapa, sy, mau, consult, dokter, malah,...",sy consult dokter ga dihubungi dokternya bayar...
1743,@mufazahd @kopites_94 Emang mau dokternya?,Emang mau dokternya?,Emang mau dokternya,emang mau dokternya,"[emang, mau, dokternya]",emang dokternya
1744,"Habis ke dokter THT, Expetasinya pengen di ber...","Habis ke dokter THT, Expetasinya pengen di ber...",Habis ke dokter THT Expetasinya pengen di bers...,habis ke dokter tht expetasinya pengen di bers...,"[habis, ke, dokter, tht, expetasinya, pengen, ...",habis dokter tht expetasinya pengen bersihin k...
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k...","Bkn cm itu, hrsnya kamera jg jgn ngeshoot dr...",Bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,"[bkn, cm, itu, hrsnya, kamera, jg, jgn, ngesho...",bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...


In [16]:
#Ambil data yang sudah dipreprocessing

data['tweet_cleans'] = data['tweet_stopwords']
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization,tweet_stopwords,tweet_cleans
0,@sagipussy Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh🤣,Anjir wkwwkwkkw dokternya auto ngeuh,anjir wkwwkwkkw dokternya auto ngeuh,"[anjir, wkwwkwkkw, dokternya, auto, ngeuh]",anjir wkwwkwkkw dokternya auto ngeuh,anjir wkwwkwkkw dokternya auto ngeuh
1,@PalingDepan__ Bukan mksdnya yg ke ruang icu n...,Bukan mksdnya yg ke ruang icu nanyain kondisi...,Bukan mksdnya yg ke ruang icu nanyain kondisi ...,bukan mksdnya yg ke ruang icu nanyain kondisi ...,"[bukan, mksdnya, yg, ke, ruang, icu, nanyain, ...",mksdnya yg ruang icu nanyain kondisi andin dok...,mksdnya yg ruang icu nanyain kondisi andin dok...
2,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis. Yaa dokternya sibuk ...,Pas lahir juga ga nangis Yaa dokternya sibuk l...,pas lahir juga ga nangis yaa dokternya sibuk l...,"[pas, lahir, juga, ga, nangis, yaa, dokternya,...",pas lahir ga nangis yaa dokternya sibuk lgi op...,pas lahir ga nangis yaa dokternya sibuk lgi op...
3,"Trs ada lagi, pas kuliah fisiologi persarafan ...","Trs ada lagi, pas kuliah fisiologi persarafan ...",Trs ada lagi pas kuliah fisiologi persarafan k...,trs ada lagi pas kuliah fisiologi persarafan k...,"[trs, ada, lagi, pas, kuliah, fisiologi, persa...",trs pas kuliah fisiologi persarafan kandung ke...,trs pas kuliah fisiologi persarafan kandung ke...
4,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ket...,dokternya bilang gigiku cantik tapi sambil ketawa,dokternya bilang gigiku cantik tapi sambil ketawa,"[dokternya, bilang, gigiku, cantik, tapi, samb...",dokternya bilang gigiku cantik ketawa,dokternya bilang gigiku cantik ketawa
...,...,...,...,...,...,...,...
1742,@HalodocID Ini kenapa sy mau consult dokter ma...,Ini kenapa sy mau consult dokter malah ga bis...,Ini kenapa sy mau consult dokter malah ga bisa...,ini kenapa sy mau consult dokter malah ga bisa...,"[ini, kenapa, sy, mau, consult, dokter, malah,...",sy consult dokter ga dihubungi dokternya bayar...,sy consult dokter ga dihubungi dokternya bayar...
1743,@mufazahd @kopites_94 Emang mau dokternya?,Emang mau dokternya?,Emang mau dokternya,emang mau dokternya,"[emang, mau, dokternya]",emang dokternya,emang dokternya
1744,"Habis ke dokter THT, Expetasinya pengen di ber...","Habis ke dokter THT, Expetasinya pengen di ber...",Habis ke dokter THT Expetasinya pengen di bers...,habis ke dokter tht expetasinya pengen di bers...,"[habis, ke, dokter, tht, expetasinya, pengen, ...",habis dokter tht expetasinya pengen bersihin k...,habis dokter tht expetasinya pengen bersihin k...
1745,"@InkaaKamal @MNC_Pictures Bkn cm itu, hrsnya k...","Bkn cm itu, hrsnya kamera jg jgn ngeshoot dr...",Bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,bkn cm itu hrsnya kamera jg jgn ngeshoot dr at...,"[bkn, cm, itu, hrsnya, kamera, jg, jgn, ngesho...",bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...


Delete the column we don't needed

In [17]:
data.drop(data.columns[[0,1,2,3,4,5]], axis = 1, inplace = True)
data

,tweet_cleans
0,anjir wkwwkwkkw dokternya auto ngeuh
1,mksdnya yg ruang icu nanyain kondisi andin dok...
2,pas lahir ga nangis yaa dokternya sibuk lgi op...
3,trs pas kuliah fisiologi persarafan kandung ke...
4,dokternya bilang gigiku cantik ketawa
...,...
1742,sy consult dokter ga dihubungi dokternya bayar...
1743,emang dokternya
1744,habis dokter tht expetasinya pengen bersihin k...
1745,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...


In [18]:
#menghapus duplikat teks
data.drop_duplicates(subset = "tweet_cleans", keep = "first", inplace=True)
data

,tweet_cleans
0,anjir wkwwkwkkw dokternya auto ngeuh
1,mksdnya yg ruang icu nanyain kondisi andin dok...
2,pas lahir ga nangis yaa dokternya sibuk lgi op...
3,trs pas kuliah fisiologi persarafan kandung ke...
4,dokternya bilang gigiku cantik ketawa
...,...
1742,sy consult dokter ga dihubungi dokternya bayar...
1743,emang dokternya
1744,habis dokter tht expetasinya pengen bersihin k...
1745,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...


**TRANSLATE TWEET CLEAN TO TWEET_ENG SO WE CAN LABEL WITH TEXTBLOB**

In [20]:
from google_trans_new import google_translator  
translator = google_translator()  

def convert_eng(tweet):
  return tweet

  translator.translate(tweet,lang_tgt='en')

data['tweet_english'] = data['tweet_cleans'].apply(convert_eng)
data

,tweet_cleans,tweet_english
0,anjir wkwwkwkkw dokternya auto ngeuh,anjir wkwwkwkkw dokternya auto ngeuh
1,mksdnya yg ruang icu nanyain kondisi andin dok...,mksdnya yg ruang icu nanyain kondisi andin dok...
2,pas lahir ga nangis yaa dokternya sibuk lgi op...,pas lahir ga nangis yaa dokternya sibuk lgi op...
3,trs pas kuliah fisiologi persarafan kandung ke...,trs pas kuliah fisiologi persarafan kandung ke...
4,dokternya bilang gigiku cantik ketawa,dokternya bilang gigiku cantik ketawa
...,...,...
1742,sy consult dokter ga dihubungi dokternya bayar...,sy consult dokter ga dihubungi dokternya bayar...
1743,emang dokternya,emang dokternya
1744,habis dokter tht expetasinya pengen bersihin k...,habis dokter tht expetasinya pengen bersihin k...
1745,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...


In [21]:
import googletrans
from googletrans import *
translator = googletrans.Translator()

data['tweet_english'] = data['tweet_english'].astype(str) #changing datatype to string
data['tweet_english'] = data['tweet_cleans'].apply(translator.translate, src='auto', dest='en').apply(getattr, args=('text',))
data

,tweet_cleans,tweet_english
0,anjir wkwwkwkkw dokternya auto ngeuh,fig wkwwkwkkw the doctor auto nodded
1,mksdnya yg ruang icu nanyain kondisi andin dok...,"I mean, the ICU room asks the doctor's conditi..."
2,pas lahir ga nangis yaa dokternya sibuk lgi op...,"When I was born, I didn't cry, the doctor was ..."
3,trs pas kuliah fisiologi persarafan kandung ke...,"then, when I studied physiology, the innervati..."
4,dokternya bilang gigiku cantik ketawa,the doctor said my teeth are beautiful laughing
...,...,...
1742,sy consult dokter ga dihubungi dokternya bayar...,"I consulted the doctor, the doctor didn't cont..."
1743,emang dokternya,really the doctor
1744,habis dokter tht expetasinya pengen bersihin k...,"after the doctor, the expectation is that he w..."
1745,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...,"not cm, the camera shouldn't shoot from above,..."


**Labeling with TextBlob**

In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

In [23]:
ps = PorterStemmer() 

def stemming_data(x):
    return ps.stem(x)

data['tweet_english'] = data['tweet_english'].apply(stemming_data)

In [24]:
data_tweet = list(data['tweet_english'])
polaritas = 0

status = []
total_positif = total_negatif = total_netral = total = 0

for i, tweet in enumerate(data_tweet):
    analysis = TextBlob(tweet)
    polaritas += analysis.polarity

    if analysis.sentiment.polarity > 0.0:
        total_positif += 1
        status.append('Positif')
    elif analysis.sentiment.polarity == 0.0:
        total_netral += 1
        status.append('Netral')
    else:
        total_negatif += 1
        status.append('Negatif')

    total += 1 
    
print(f'Hasil Analisis Data:\nPositif = {total_positif}\nNetral = {total_netral}\nNegatif = {total_negatif}')
print(f'\nTotal Data : {total}')

Hasil Analisis Data:
Positif = 568
Netral = 573
Negatif = 429

Total Data : 1570


In [25]:
status = pd.DataFrame({'klasifikasi': status})
data['klasifikasi'] = status
data.tail()

,tweet_cleans,tweet_english,klasifikasi
1742,sy consult dokter ga dihubungi dokternya bayar...,"i consulted the doctor, the doctor didn't cont...",NaN
1743,emang dokternya,really the doctor,NaN
1744,habis dokter tht expetasinya pengen bersihin k...,"after the doctor, the expectation is that he w...",NaN
1745,bkn cm hrsnya kamera jgn ngeshoot dr ats ketau...,"not cm, the camera shouldn't shoot from above,...",NaN
1746,disaranin terapi psikolog but trauma gara psik...,advised psychological therapy but trauma becau...,NaN


**SIMPAN DATASET KE FILE EXCEL DI DRIVE KITA**

In [27]:
data.to_excel('./dataset-testing-preprocessing.xlsx', encoding='utf8', index=True)